# Synthetic Control with Prediction Intervals: Quantifying Uncertainty in Germany's Reunification Impact

When a policy affects an entire country, there is no untreated twin to compare it against. The **synthetic control method** addresses this challenge by constructing an artificial counterfactual --- a weighted combination of similar units that mimics what the treated unit would have looked like without the intervention. Introduced by Abadie, Diamond, and Hainmueller (2010, 2015), this approach has become one of the most widely used tools in comparative case studies.

Yet the classic synthetic control delivers only a **point estimate**. Researchers see a gap between the treated unit and its synthetic counterpart, but they have no formal way to judge whether that gap reflects a real policy effect or just noise. Placebo tests offer suggestive evidence, but they do not produce confidence intervals with well-defined coverage guarantees.

Cattaneo, Feng, and Titiunik (2021) solve this problem by developing **prediction intervals for synthetic control methods**. Their key insight is that uncertainty comes from two distinct sources: (1) the weights themselves are estimated from a finite pre-treatment sample, and (2) the post-treatment world may deviate from the model in ways that pre-treatment data cannot predict. By quantifying both sources separately, the SCPI framework produces intervals with finite-sample coverage guarantees --- not just asymptotic approximations.

In this tutorial, we apply the SCPI framework to a classic question in political economy: **Did German reunification in 1990 reduce West Germany's GDP per capita, and how confident can we be in that estimate?** Using GDP data for 17 countries from 1960 to 2003, we construct a synthetic West Germany, estimate the treatment effect, and --- crucially --- build prediction intervals that tell us whether the effect is statistically distinguishable from zero.

**Learning objectives:**

- Understand the logic of synthetic control: constructing a counterfactual from weighted donor units
- Implement point estimation and prediction intervals using the Python [`scpi_pkg`](https://nppackages.github.io/scpi/) package
- Distinguish the two sources of uncertainty in synthetic control predictions: in-sample (weight estimation) and out-of-sample (post-treatment misspecification)
- Construct and interpret prediction intervals with finite-sample coverage guarantees
- Compare alternative weight constraint methods (simplex, lasso, ridge, OLS) and assess their trade-offs
- Evaluate robustness through sensitivity analysis across confidence levels

## The Synthetic Control Idea

The core intuition behind synthetic control is straightforward. Imagine you want to know how reunification changed West Germany's economic trajectory. You cannot simply compare West Germany's GDP after 1990 to its GDP before 1990, because many other factors --- global recessions, trade liberalization, technological change --- also affected the economy over that period.

Instead, you build a **synthetic West Germany**: a weighted average of other countries that, collectively, track West Germany's GDP trajectory closely during the pre-reunification period (1960--1990). If the synthetic version continues along a plausible path after 1990 while the actual West Germany diverges, the gap measures the causal effect of reunification.

Think of it as building a custom control group from scratch. Rather than picking a single comparison country (which might differ from West Germany in important ways), you blend multiple countries together so that their weighted average resembles West Germany as closely as possible --- like mixing paints to match a target color.

**The estimation pipeline:**

1. **West Germany (treated unit)** → Pre-treatment GDP 1960--1990
2. **16 Donor Countries (control pool)** → Find weights $w_1 \ldots w_{16}$ to match pre-treatment GDP
3. **Synthetic West Germany** → Post-1990 gap = Treatment effect $\tau$

Formally, the treatment effect at each post-treatment period $T$ is the difference between what we observe and the counterfactual:

$$\tau_T = Y_{1T}(1) - Y_{1T}(0)$$

In words, this equation says that the treatment effect $\tau_T$ equals the observed outcome $Y_{1T}(1)$ minus the counterfactual outcome $Y_{1T}(0)$ --- what West Germany's GDP would have been without reunification. Since we cannot observe $Y_{1T}(0)$ directly, we estimate it using the synthetic control.

The synthetic counterfactual prediction is a weighted sum of donor outcomes:

$$\hat{Y}_{1T}(0) = \mathbf{x}_T' \hat{\mathbf{w}}$$

Here, $\mathbf{x}_T$ is the vector of donor country GDP values at time $T$, and $\hat{\mathbf{w}}$ is the vector of estimated weights. In the classic formulation, these weights are non-negative and sum to one, ensuring the synthetic control is a convex combination of real countries.

## Why Point Estimates Are Not Enough

The classic synthetic control gives us a single number --- the estimated gap --- but no formal measure of how precise that estimate is. Cattaneo, Feng, and Titiunik (2021) show that this uncertainty comes from two separate sources, and both must be accounted for:

$$\hat{\tau}_T - \tau_T = \underbrace{\mathbf{p}_T'(\boldsymbol{\beta}_0 - \hat{\boldsymbol{\beta}})}_{\text{in-sample}} + \underbrace{e_T}_{\text{out-of-sample}}$$

In words, this equation says that the error in our treatment effect estimate has two components. The first term, called **in-sample uncertainty**, arises because we estimate the weights $\hat{\boldsymbol{\beta}}$ from a finite number of pre-treatment periods. With only 31 years of data to estimate 16 weights, there is inherent sampling variability. The true best-fitting weights $\boldsymbol{\beta}_0$ may differ from our estimates, and this difference propagates into the post-treatment prediction through the post-treatment data $\mathbf{p}_T$.

The second term, **out-of-sample uncertainty** ($e_T$), captures everything that the model cannot predict from pre-treatment data alone. Even if we knew the perfect weights, the post-reunification world might generate shocks --- structural breaks, unforeseen economic events --- that push the actual counterfactual away from our weighted prediction. This is analogous to forecasting: even the best model has a prediction error when projecting into the future.

The SCPI framework constructs prediction intervals that account for both sources simultaneously. By bounding each component separately and combining them, the resulting intervals carry **finite-sample coverage guarantees** --- they contain the true treatment effect with at least the stated probability, without relying on large-sample approximations.

## Setup and Data

We use the [`scpi_pkg`](https://nppackages.github.io/scpi/) Python package, which implements the methods from Cattaneo, Feng, and Titiunik (2021). The package provides four core functions: [`scdata()`](https://nppackages.github.io/scpi/reference/scdata.html) for data preparation, [`scest()`](https://nppackages.github.io/scpi/reference/scest.html) for point estimation, [`scpi()`](https://nppackages.github.io/scpi/reference/scpi.html) for prediction intervals, and [`scplot()`](https://nppackages.github.io/scpi/reference/scplot.html) for visualization.

Before running the analysis, install the required package:

In [ ]:
!pip install -q scpi_pkg

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Adapted from scpi_pkg illustration scripts:
# https://github.com/nppackages/scpi/tree/main/Python/scpi_illustration
from scpi_pkg.scdata import scdata
from scpi_pkg.scest import scest
from scpi_pkg.scpi import scpi

# Reproducibility
RANDOM_SEED = 8894
np.random.seed(RANDOM_SEED)

# ── Site color palette ──────────────────────────────────────────────
STEEL_BLUE = "#6a9bcc"
WARM_ORANGE = "#d97757"
NEAR_BLACK = "#141413"
TEAL = "#00d4c8"

# Dark theme palette
DARK_NAVY = "#0f1729"
GRID_LINE = "#1f2b5e"
LIGHT_TEXT = "#c8d0e0"
WHITE_TEXT = "#e8ecf2"

# Plot defaults — minimal, spine-free, dark background
plt.rcParams.update({
    "figure.facecolor": DARK_NAVY,
    "axes.facecolor": DARK_NAVY,
    "axes.edgecolor": DARK_NAVY,
    "axes.linewidth": 0,
    "axes.labelcolor": LIGHT_TEXT,
    "axes.titlecolor": WHITE_TEXT,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.spines.bottom": False,
    "axes.grid": True,
    "grid.color": GRID_LINE,
    "grid.linewidth": 0.6,
    "grid.alpha": 0.8,
    "xtick.color": LIGHT_TEXT,
    "ytick.color": LIGHT_TEXT,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "text.color": WHITE_TEXT,
    "font.size": 12,
    "legend.frameon": False,
    "legend.fontsize": 11,
    "legend.labelcolor": LIGHT_TEXT,
    "figure.edgecolor": DARK_NAVY,
    "savefig.facecolor": DARK_NAVY,
    "savefig.edgecolor": DARK_NAVY,
})

The dataset contains GDP per capita (in thousands of US dollars) for 17 countries from 1960 to 2003. West Germany is the treated unit, and the remaining 16 countries form the donor pool. The data is sourced from Abadie (2021), who used it to study the economic consequences of reunification.

In [ ]:
# Download the Germany reunification dataset
# Source: Abadie, Diamond, and Hainmueller (2015)
try:
    data = pd.read_csv(
        "https://raw.githubusercontent.com/nppackages/scpi/main/Python/scpi_illustration/scpi_germany.csv"
    )
except Exception:
    # Alternative: upload data.csv from your local machine
    from google.colab import files
    print("Auto-download failed. Please upload data.csv:")
    uploaded = files.upload()
    data = pd.read_csv("data.csv")

print(f"Shape: {data.shape}")
print(f"Countries ({data['country'].nunique()}):")
print(sorted(data['country'].unique()))
print(f"\nYear range: {data['year'].min()} \u2013 {data['year'].max()}")
print(f"\nGDP per capita (thousand USD):")
print(data['gdp'].describe().round(3))

The dataset covers 748 observations across 17 countries and 44 years. GDP per capita ranges from \$707 (Portugal, early 1960s) to \$37,548 (Norway, early 2000s), with a mean of \$12,144. West Germany sits in the upper portion of this distribution, which means the synthetic control will need to weight richer countries more heavily. The panel is well suited for synthetic control analysis because it provides 31 pre-treatment years --- a substantial window for estimating donor weights accurately.

## Exploring the Data

Before building a synthetic control, it helps to visualize how West Germany's GDP trajectory compares to the donor pool. This reveals whether reunification produced a visible divergence and which countries might serve as good donors.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)

countries = sorted(data['country'].unique())
for country in countries:
    cdata = data[data['country'] == country]
    if country == 'West Germany':
        ax.plot(cdata['year'], cdata['gdp'], color=WARM_ORANGE, linewidth=2.5,
                label='West Germany', zorder=10)
    else:
        ax.plot(cdata['year'], cdata['gdp'], color=STEEL_BLUE, alpha=0.3,
                linewidth=1)

ax.axvline(x=1990, color=TEAL, linestyle='--', linewidth=1.5, alpha=0.8,
           label='Reunification (1990)')
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('GDP per Capita (thousand USD)', fontsize=13)
ax.set_title('GDP Trajectories: West Germany vs. Donor Pool', fontsize=15, pad=12)
ax.legend(loc='upper left', fontsize=11)
plt.savefig("scpi_gdp_trajectories.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

West Germany's GDP (orange line) grows steadily from about \$2,300 in 1960 to \$20,500 by 1990, tracking closely with the upper cluster of industrialized nations. After reunification in 1990, the growth trajectory appears to flatten relative to several donor countries that continue climbing. This visual impression of slower post-reunification growth is exactly what the synthetic control method will test formally. The key question is whether this flattening is statistically significant or could be explained by normal economic variation across countries.

## Preparing the Data for SCPI

The [`scdata()`](https://nppackages.github.io/scpi/reference/scdata.html) function structures the panel into the format required for estimation. We define the treatment period (reunification in 1991), the pre-treatment window (1960--1990), and the donor pool. The `cointegrated_data=True` flag tells the estimator that GDP series are likely non-stationary --- they trend upward over time --- so the method adjusts for common stochastic trends rather than assuming stationarity.

In [ ]:
id_var = 'country'
outcome_var = 'gdp'
time_var = 'year'
period_pre = np.arange(1960, 1991)   # 1960–1990 (31 years)
period_post = np.arange(1991, 2004)  # 1991–2003 (13 years)
unit_tr = 'West Germany'
unit_co = [c for c in sorted(data[id_var].unique()) if c != unit_tr]

print(f"Treated unit: {unit_tr}")
print(f"Donor pool ({len(unit_co)} countries): {unit_co}")
print(f"Pre-treatment period: {period_pre[0]}\u2013{period_pre[-1]} ({len(period_pre)} years)")
print(f"Post-treatment period: {period_post[0]}\u2013{period_post[-1]} ({len(period_post)} years)")

data_prep = scdata(df=data, id_var=id_var, time_var=time_var,
                   outcome_var=outcome_var, period_pre=period_pre,
                   period_post=period_post, unit_tr=unit_tr,
                   unit_co=unit_co, features=None, cov_adj=None,
                   cointegrated_data=True, constant=False)

The prepared data object contains 31 pre-treatment observations per country and 13 post-treatment observations. With 16 donor countries available, the simplex constraint (weights summing to one) ensures a well-defined convex combination. Setting `cointegrated_data=True` is important here because GDP series share a common upward trend driven by global economic growth, and treating them as stationary would distort the weight estimation.

## Point Estimation: Building Synthetic West Germany

The [`scest()`](https://nppackages.github.io/scpi/reference/scest.html) function estimates the donor weights by minimizing the pre-treatment prediction error. With `w_constr={'name': 'simplex'}`, we impose the classic constraint: weights must be non-negative and sum to one. This means the synthetic West Germany is a convex combination of real countries --- no extrapolation beyond the donor pool's range.

In [ ]:
est_si = scest(data_prep, w_constr={'name': "simplex"})
print(est_si)

The estimator selects 6 out of 16 donor countries, assigning zero weight to the remaining 10. Austria receives the largest weight (0.291), followed by the USA (0.273), Italy (0.191), the Netherlands (0.133), Switzerland (0.081), and France (0.030). The selection makes economic sense: Austria shares a border, language, and institutional history with West Germany; the USA and Italy are large economies that tracked similar growth patterns during this period. Countries like Greece, Portugal, and Spain --- which had significantly lower GDP levels and different growth trajectories --- receive zero weight, as including them would worsen the pre-treatment fit.

In [ ]:
y_pre_actual = est_si.Y_pre.values.flatten()
y_post_actual = est_si.Y_post.values.flatten()
y_pre_fit = est_si.Y_pre_fit.values.flatten()
y_post_fit = est_si.Y_post_fit.values.flatten()

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)

# Bridge pre/post by concatenating for continuous lines
time_actual = np.concatenate([period_pre, period_post])
y_all_actual = np.concatenate([y_pre_actual, y_post_actual])
y_all_fit = np.concatenate([y_pre_fit, y_post_fit])

ax.plot(time_actual, y_all_actual, color=WARM_ORANGE, linewidth=2.2,
        label='West Germany (actual)')
ax.plot(time_actual, y_all_fit, color=STEEL_BLUE, linewidth=2.2,
        linestyle='--', label='Synthetic West Germany')
ax.axvline(x=1990, color=TEAL, linestyle='--', linewidth=1.5, alpha=0.8,
           label='Reunification (1990)')

ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('GDP per Capita (thousand USD)', fontsize=13)
ax.set_title('Actual vs. Synthetic West Germany', fontsize=15, pad=12)
ax.legend(loc='upper left', fontsize=11)
plt.savefig("scpi_actual_vs_synthetic.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

The synthetic West Germany (blue dashed line) tracks the actual trajectory (orange solid line) nearly perfectly throughout the pre-treatment period, confirming that the donor weights produce a credible counterfactual. After reunification in 1990, the two lines diverge: the synthetic version continues climbing at the pre-reunification pace, while actual West Germany's growth slows noticeably. By 2003, the gap between the two series is visually substantial. This pre-treatment fit is crucial --- if the synthetic control could not match the treated unit before the intervention, we would have little reason to trust its post-treatment predictions.

### Examining the Weights

To understand which countries drive the synthetic control, we can visualize the estimated weights directly. This reveals the composition of our counterfactual West Germany.

In [ ]:
w_df = est_si.w.copy()
w_df.columns = ['weight']
w_df = w_df[w_df['weight'] > 0.001].sort_values('weight', ascending=True)
print(w_df.round(4))
print(f"\nCountries with non-zero weight: {len(w_df)}")

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_linewidth(0)

countries_w = [idx[1] if isinstance(idx, tuple) and len(idx) > 1 else idx for idx in w_df.index]
weights = w_df['weight'].values

bars = ax.barh(range(len(countries_w)), weights, color=STEEL_BLUE,
               edgecolor=DARK_NAVY, height=0.6)

# Highlight the largest weight
max_idx = np.argmax(weights)
bars[max_idx].set_color(WARM_ORANGE)

ax.set_yticks(range(len(countries_w)))
ax.set_yticklabels(countries_w, fontsize=11)
ax.set_xlabel('Weight', fontsize=13)
ax.set_title('Synthetic Control Weights (Simplex)', fontsize=15, pad=12)

for i, w in enumerate(weights):
    ax.text(w + 0.005, i, f'{w:.3f}', va='center', fontsize=10, color=LIGHT_TEXT)

plt.savefig("scpi_weights.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

Austria and the USA together account for over 56% of the synthetic West Germany, reflecting their dominant role in replicating the treated unit's economic trajectory. The remaining weight is split among four Western European economies. The sparsity of the solution --- only 6 of 16 countries receiving positive weight --- is a feature, not a limitation. Sparse weights make the counterfactual more interpretable: synthetic West Germany is primarily a blend of Austria, the USA, and Italy, rather than a diffuse average across all donors.

### The Estimated Treatment Effect

The treatment effect in each post-reunification year is simply the gap between actual and synthetic GDP. A negative gap means reunification reduced West Germany's GDP relative to what the synthetic counterfactual predicts.

In [ ]:
gap_post = y_post_actual - y_post_fit
gap_df = pd.DataFrame({
    'Year': period_post,
    'Actual': y_post_actual.round(3),
    'Synthetic': y_post_fit.round(3),
    'Gap': gap_post.round(3)
})
print(gap_df.to_string(index=False))
print(f"\nAverage gap (1991\u20132003): {gap_post.mean():.3f} thousand USD")
print(f"Gap in 2003 (final year): {gap_post[-1]:.3f} thousand USD")

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_linewidth(0)

ax.bar(period_post, gap_post,
       color=[WARM_ORANGE if g < 0 else TEAL for g in gap_post],
       edgecolor=DARK_NAVY, width=0.7)
ax.axhline(y=0, color=LIGHT_TEXT, linewidth=0.8, alpha=0.5)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Gap (Actual - Synthetic, thousand USD)', fontsize=13)
ax.set_title('Estimated Treatment Effect of Reunification', fontsize=15, pad=12)
plt.savefig("scpi_treatment_gap.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

The gap starts small and positive in 1991--1992 (\$502 and \$325), suggesting a brief initial boost or delayed onset. By 1993, the effect turns negative and grows steadily: from -\$440 in 1993 to -\$3,465 in 2003. The average gap over the entire post-reunification period is -\$1,668 thousand per capita. In practical terms, by 2003 West Germany's GDP per capita was approximately \$3,500 lower than what the synthetic control predicts it would have been without reunification --- a substantial and growing economic cost. However, these are point estimates with no uncertainty measure attached. The crucial question remains: could this gap be explained by normal cross-country variation? That is exactly what prediction intervals address.

## Prediction Intervals: Quantifying Uncertainty

The [`scpi()`](https://nppackages.github.io/scpi/reference/scpi.html) function extends point estimation by constructing prediction intervals that account for both in-sample and out-of-sample uncertainty. The function uses Monte Carlo simulation to approximate the distribution of the in-sample component and a Gaussian concentration inequality for the out-of-sample component.

Key parameters control how the uncertainty is modeled:
- `u_missp=True` allows for model misspecification in the in-sample component, making the intervals more conservative and realistic
- `u_sigma="HC1"` uses heteroskedasticity-consistent variance estimation, which is robust to non-constant variance across time periods
- `e_method="gaussian"` assumes sub-Gaussian tails for the out-of-sample error, providing tight but reliable bounds
- `sims=200` sets the number of Monte Carlo replications for approximating the in-sample distribution

In [ ]:
w_constr = {'name': 'simplex', 'Q': 1}
pi_si = scpi(data_prep, sims=200, w_constr=w_constr,
             u_order=1, u_lags=0,
             e_order=1, e_lags=0,
             e_method="gaussian",
             u_missp=True, u_sigma="HC1",
             cores=1, e_alpha=0.05, u_alpha=0.05)
print(pi_si)

The prediction intervals show the range within which the synthetic control estimate (the counterfactual GDP) is expected to fall with 95% probability. What matters is whether the **actual** West Germany GDP falls inside or outside these intervals. Looking at the results, the actual GDP (Treated column) falls **below the lower bound** of the prediction interval for nearly every year from 1997 onward. For example, in 2003 the actual GDP is 28.86 while the lower bound of the PI is 30.04 --- actual GDP is \$1,180 below even the most conservative prediction. This means the negative treatment effect is statistically significant: the gap cannot be explained by estimation uncertainty or normal post-treatment variation alone.

### Visualizing the Prediction Intervals

A plot makes the significance pattern immediately clear. When the actual GDP line falls outside the shaded prediction interval band, the treatment effect is statistically distinguishable from zero at the 95% level.

In [ ]:
ci_all = pi_si.CI_all_gaussian
ci_lower = ci_all.iloc[:, 0].values
ci_upper = ci_all.iloc[:, 1].values
ci_years = ci_all.index.get_level_values(1).tolist()

# Pre-treatment fitted values from PI object
y_pre_fit_pi = pi_si.Y_pre_fit.values.flatten()
y_post_fit_pi = pi_si.Y_post_fit.values.flatten()
y_pre_actual_pi = pi_si.Y_pre.values.flatten()
y_post_actual_pi = pi_si.Y_post.values.flatten()

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)

# Continuous lines bridging pre/post
time_all = np.concatenate([period_pre, period_post])
ax.plot(time_all, np.concatenate([y_pre_actual_pi, y_post_actual_pi]),
        color=WARM_ORANGE, linewidth=2.2, label='West Germany (actual)')
ax.plot(time_all, np.concatenate([y_pre_fit_pi, y_post_fit_pi]),
        color=STEEL_BLUE, linewidth=2.2, linestyle='--',
        label='Synthetic West Germany')

# Anchor PI band at last pre-treatment point (1990) where uncertainty ~ 0
last_pre_fit = y_pre_fit_pi[-1]
pi_time = np.concatenate([[period_pre[-1]], period_post])
pi_lower = [last_pre_fit]
pi_upper = [last_pre_fit]
for yr in period_post:
    if yr in ci_years:
        idx = ci_years.index(yr)
        pi_lower.append(ci_lower[idx])
        pi_upper.append(ci_upper[idx])
    else:
        pi_lower.append(np.nan)
        pi_upper.append(np.nan)

ax.fill_between(pi_time, pi_lower, pi_upper,
                color=STEEL_BLUE, alpha=0.2, label='95% Prediction Interval')

ax.axvline(x=1990, color=TEAL, linestyle='--', linewidth=1.5, alpha=0.8,
           label='Reunification (1990)')
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('GDP per Capita (thousand USD)', fontsize=13)
ax.set_title('Synthetic Control with Prediction Intervals', fontsize=15, pad=12)
ax.legend(loc='upper left', fontsize=10)
plt.savefig("scpi_prediction_intervals.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

The shaded band represents the 95% prediction interval for the synthetic control's counterfactual GDP. In the early post-reunification years (1991--1996), the actual GDP (orange line) sits near or just below the lower edge of the band, suggesting the effect is emerging but not yet statistically significant at the 95% level. From 1997 onward, actual GDP falls clearly below the prediction interval, and the gap widens each year. By 2003, West Germany's actual GDP of \$28,855 sits nearly \$1,200 below the lower bound of \$30,040. This pattern tells a clear story: the economic cost of reunification was not just a short-term shock but a persistent structural drag that became statistically unmistakable within a decade.

## Robustness: Alternative Weight Constraints

The classic simplex constraint (non-negative weights summing to one) is the standard choice, but it is not the only option. The `scpi_pkg` supports several alternatives. Each imposes different assumptions on the weight structure, and comparing their results reveals how sensitive our conclusions are to these modeling choices.

- **Simplex** (classic SC): Weights are non-negative and sum to one. Produces an interpretable convex combination of donors. Most constrained.
- **Lasso**: Weights sum to at most one in absolute value. Encourages sparsity --- like simplex, but allows some weights to shrink to zero more aggressively.
- **Ridge**: Weights are penalized by their L2 norm. Allows all donors to contribute small weights, reducing variance at the cost of some bias.
- **OLS**: No constraints on weights. Least restrictive --- weights can be negative or exceed one. Most flexible, but risks extrapolation beyond the donor range.

In [ ]:
est_lasso = scest(data_prep, w_constr={'name': "lasso"})
est_ridge = scest(data_prep, w_constr={'name': "ridge"})
est_ls = scest(data_prep, w_constr={'name': "ols"})

methods = {
    'Simplex': est_si,
    'Lasso': est_lasso,
    'Ridge': est_ridge,
    'OLS': est_ls
}

print(f"{'Method':<12} {'Pre-RMSE':<12} {'Gap 2003':<12} {'Avg Gap':<12}")
print("-" * 48)
for name, est in methods.items():
    pre_resid = est.Y_pre.values.flatten() - est.Y_pre_fit.values.flatten()
    pre_rmse = np.sqrt(np.mean(pre_resid**2))
    post_gap = est.Y_post.values.flatten() - est.Y_post_fit.values.flatten()
    print(f"{name:<12} {pre_rmse:<12.3f} {post_gap[-1]:<12.3f} {post_gap.mean():<12.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_linewidth(0)

colors_methods = {
    'Simplex': STEEL_BLUE,
    'Lasso': WARM_ORANGE,
    'Ridge': TEAL,
    'OLS': LIGHT_TEXT
}

for ax, (name, est) in zip(axes.flatten(), methods.items()):
    color = colors_methods[name]

    y_pre_a = est.Y_pre.values.flatten()
    y_post_a = est.Y_post.values.flatten()
    y_pre_f = est.Y_pre_fit.values.flatten()
    y_post_f = est.Y_post_fit.values.flatten()

    t_all = np.concatenate([period_pre, period_post])
    ax.plot(t_all, np.concatenate([y_pre_a, y_post_a]),
            color=WARM_ORANGE, linewidth=1.8, label='Actual')
    ax.plot(t_all, np.concatenate([y_pre_f, y_post_f]),
            color=color, linewidth=1.8, linestyle='--',
            label=f'Synthetic ({name})')
    ax.axvline(x=1990, color=TEAL if name != 'Ridge' else WARM_ORANGE,
               linestyle=':', linewidth=1, alpha=0.6)

    pre_resid = y_pre_a - y_pre_f
    pre_rmse = np.sqrt(np.mean(pre_resid**2))
    post_gap = y_post_a - y_post_f
    ax.set_title(f'{name}  (Pre-RMSE: {pre_rmse:.2f}, Gap 2003: {post_gap[-1]:.1f})',
                 fontsize=12, pad=8)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_xlabel('Year', fontsize=10)
    ax.set_ylabel('GDP per Capita', fontsize=10)

fig.suptitle('Comparing Weight Constraint Methods', fontsize=16, y=1.01,
             color=WHITE_TEXT)
plt.tight_layout()
plt.savefig("scpi_method_comparison.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

| Method | Pre-RMSE | Gap in 2003 | Average Gap |
|--------|----------|-------------|-------------|
| Simplex | 0.072 | -3.465 | -1.668 |
| Lasso | 0.071 | -3.426 | -1.618 |
| Ridge | 0.040 | -2.719 | -1.415 |
| OLS | 0.040 | -2.380 | -1.323 |

All four methods agree on the direction and general magnitude of the effect: reunification reduced West Germany's GDP per capita. The simplex and lasso constraints produce nearly identical results (pre-RMSE of 0.072 and 0.071, gap in 2003 of -\$3,465 and -\$3,426), which is expected since lasso is a relaxation of simplex. Ridge and OLS achieve a tighter pre-treatment fit (RMSE of 0.040) by allowing more flexible weights, but they estimate a somewhat smaller gap (-\$2,719 and -\$2,380 in 2003). The smaller gap under OLS is typical: unconstrained weights can overfit the pre-treatment period, which slightly reduces the apparent post-treatment divergence. The key takeaway is that the negative treatment effect is robust across all weight specifications --- the choice of constraint affects magnitude but not the qualitative conclusion.

## Sensitivity Analysis

How sensitive are the prediction intervals to the confidence level? Wider intervals (higher confidence) are harder to reject, so checking whether the actual GDP falls outside the band at multiple confidence levels reveals how robust the statistical significance is.

In [ ]:
alphas = [0.01, 0.05, 0.10, 0.20]
sensitivity_results = {}

for alpha in alphas:
    np.random.seed(RANDOM_SEED)
    pi_temp = scpi(data_prep, sims=200, w_constr={'name': 'simplex', 'Q': 1},
                   u_order=1, u_lags=0, e_order=1, e_lags=0,
                   e_method="gaussian", u_missp=True, u_sigma="HC1",
                   cores=1, e_alpha=alpha, u_alpha=alpha)
    ci_temp = pi_temp.CI_all_gaussian
    sensitivity_results[alpha] = {
        'lower': ci_temp.iloc[:, 0].values,
        'upper': ci_temp.iloc[:, 1].values,
        'years': ci_temp.index.get_level_values(1).tolist()
    }

print(f"{'Alpha':<10} {'Coverage':<12} {'Avg PI Width':<15}")
print("-" * 37)
for alpha, res in sensitivity_results.items():
    post_years_in_ci = [yr for yr in period_post if yr in res['years']]
    inside = 0
    total = 0
    post_widths = []
    for yr in post_years_in_ci:
        yr_idx_ci = res['years'].index(yr)
        yr_idx_post = list(period_post).index(yr)
        if res['lower'][yr_idx_ci] <= y_post_actual_pi[yr_idx_post] <= res['upper'][yr_idx_ci]:
            inside += 1
        total += 1
        post_widths.append(res['upper'][yr_idx_ci] - res['lower'][yr_idx_ci])
    coverage = f"{inside}/{total}"
    avg_width = np.mean(post_widths) if post_widths else 0
    print(f"{1-alpha:<10.0%} {coverage:<12} {avg_width:<15.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_linewidth(0)

colors_alpha = {0.01: TEAL, 0.05: STEEL_BLUE, 0.10: WARM_ORANGE, 0.20: LIGHT_TEXT}
labels_alpha = {0.01: '99% PI', 0.05: '95% PI', 0.10: '90% PI', 0.20: '80% PI'}

# Continuous lines bridging pre/post
t_all = np.concatenate([period_pre, period_post])
ax.plot(t_all, np.concatenate([y_pre_actual_pi, y_post_actual_pi]),
        color=WARM_ORANGE, linewidth=2, label='West Germany (actual)')
ax.plot(t_all, np.concatenate([y_pre_fit_pi, y_post_fit_pi]),
        color=STEEL_BLUE, linewidth=1.5, linestyle='--', alpha=0.5,
        label='Synthetic')

# Overlay PI bands from widest to narrowest, anchored at 1990
last_pre_fit_sens = y_pre_fit_pi[-1]
pi_time_sens = np.concatenate([[period_pre[-1]], period_post])
for alpha in [0.01, 0.05, 0.10, 0.20]:
    res = sensitivity_results[alpha]
    band_lower = [last_pre_fit_sens]
    band_upper = [last_pre_fit_sens]
    for yr in period_post:
        if yr in res['years']:
            idx = res['years'].index(yr)
            band_lower.append(res['lower'][idx])
            band_upper.append(res['upper'][idx])
        else:
            band_lower.append(np.nan)
            band_upper.append(np.nan)

    ax.fill_between(pi_time_sens, band_lower, band_upper,
                    color=colors_alpha[alpha], alpha=0.15,
                    label=labels_alpha[alpha])

ax.axvline(x=1990, color=TEAL, linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('GDP per Capita (thousand USD)', fontsize=13)
ax.set_title('Sensitivity Analysis: Prediction Intervals at Different Confidence Levels',
             fontsize=14, pad=12)
ax.legend(loc='upper left', fontsize=9)
plt.savefig("scpi_sensitivity.png", dpi=300, bbox_inches="tight",
            facecolor=DARK_NAVY, edgecolor=DARK_NAVY, pad_inches=0)
plt.show()

Even with the widest 99% prediction intervals (average width of \$3,298 thousand), actual West Germany GDP falls outside the band for 7 of the 13 post-treatment years. At the 90% level, it falls outside for 9 of 13 years. The pattern is clear: the economic impact of reunification is robust to the choice of confidence level. For the final years of the sample (roughly 1997--2003), actual GDP lies below **all four** PI bands simultaneously, confirming that the negative effect is highly statistically significant. A researcher would need to assume implausibly large out-of-sample uncertainty to overturn this conclusion.

## Discussion

Returning to our original question: **Did German reunification reduce West Germany's GDP per capita?** The evidence strongly supports a negative and persistent effect. The synthetic control estimates show that by 2003, West Germany's GDP per capita was approximately \$3,465 lower than what the synthetic counterfactual predicts --- a gap that grew steadily from near zero in 1991 to over \$3,000 by the early 2000s.

Crucially, the SCPI prediction intervals confirm this effect is **statistically significant**. From the mid-1990s onward, actual GDP falls below the lower bound of the 95% prediction interval, and this pattern holds even at the 99% confidence level. The sensitivity analysis shows that the conclusion is robust: no reasonable assumption about out-of-sample uncertainty can explain away the gap.

For policymakers, the finding highlights that large-scale political integration --- even between regions that share a language and cultural heritage --- can impose substantial and long-lasting economic costs on the wealthier partner. West Germany effectively subsidized the reconstruction of the East German economy, and these transfers show up as a persistent drag on per capita GDP. The magnitude --- roughly \$3,500 per person by 2003, or about 11% of predicted GDP --- represents a significant reallocation of economic resources.

These results align with Abadie (2021), who reached similar qualitative conclusions using the classic synthetic control method. The contribution of the SCPI framework is to move beyond point estimates and provide formal uncertainty quantification, transforming an informal visual assessment ("the lines diverge") into a rigorous statistical statement ("the gap exceeds what can be explained by estimation or prediction uncertainty").

## Summary and Next Steps

**Key takeaways:**

1. **Method insight.** The synthetic control method is particularly powerful when only one unit receives a treatment and traditional difference-in-differences designs are not feasible. The SCPI extension solves a longstanding limitation by providing prediction intervals with finite-sample coverage guarantees, decomposing uncertainty into in-sample (weight estimation) and out-of-sample (post-treatment shocks) components.

2. **Data insight.** Six of sixteen donor countries receive positive weight in the synthetic West Germany, led by Austria (0.291), the USA (0.273), and Italy (0.191). The pre-treatment RMSE of 0.072 confirms an excellent fit, and the gap grows from near zero in 1991 to -\$3,465 by 2003.

3. **Practical limitation.** The synthetic control method assumes that the donor pool contains countries whose weighted combination can approximate the treated unit's trajectory. If the treated unit is fundamentally different from all available donors --- or if the intervention changes the relationships between the treated unit and its donors --- the counterfactual may be unreliable. Additionally, the method cannot account for spillover effects: reunification may have affected the donor countries themselves through trade and migration channels.

4. **Next step.** The `scpi_pkg` package supports multiple treated units via [`scdataMulti()`](https://nppackages.github.io/scpi/reference/scdataMulti.html), enabling staggered adoption designs. Readers interested in extensions could also experiment with covariate adjustment (adding trade openness or inflation as matching features) or alternative PI methods (location-scale and quantile regression) to compare with the Gaussian bounds used here.

**Limitations:**

- Results depend on the donor pool composition. Excluding or including specific countries can shift the estimated gap.
- The cointegrated data setting assumes a shared stochastic trend across countries; if this assumption fails, weights may be biased.
- With only one treated unit, we cannot assess heterogeneity in treatment effects across different types of reunification scenarios.

## Exercises

1. **Add covariates.** Re-run the analysis with `features=['gdp', 'trade']` in `scdata()`. Does matching on trade openness in addition to GDP change the estimated weights or the treatment effect?

2. **Modify the donor pool.** Remove Austria and the USA (the two highest-weighted donors) and re-estimate. How sensitive is the gap to the composition of the donor pool?

3. **Alternative PI method.** Replace `e_method="gaussian"` with `e_method="ls"` (location-scale) in `scpi()`. Compare the width and shape of the resulting prediction intervals. Under what conditions would you prefer one method over the other?

## References

1. [Cattaneo, M. D., Feng, Y., and Titiunik, R. (2021). Prediction Intervals for Synthetic Control Methods. *Journal of the American Statistical Association*, 116(536), 1668--1683.](https://doi.org/10.1080/01621459.2021.1979561)
2. [Abadie, A., Diamond, A., and Hainmueller, J. (2015). Comparative Politics and the Synthetic Control Method. *American Journal of Political Science*, 59(2), 495--510.](https://doi.org/10.1111/ajps.12116)
3. [Abadie, A., Diamond, A., and Hainmueller, J. (2010). Synthetic Control Methods for Comparative Case Studies: Estimating the Effect of California's Tobacco Control Program. *Journal of the American Statistical Association*, 105(490), 493--505.](https://doi.org/10.1198/jasa.2009.ap08746)
4. [Abadie, A. (2021). Using Synthetic Controls: Feasibility, Data Requirements, and Methodological Aspects. *Journal of Economic Literature*, 59(2), 391--425.](https://doi.org/10.1257/jel.20191450)
5. [scpi_pkg --- Python package for Synthetic Control with Prediction Intervals.](https://nppackages.github.io/scpi/)
6. [Cattaneo, M. D., Feng, Y., Palomba, F., and Titiunik, R. scpi_pkg illustration scripts (GitHub).](https://github.com/nppackages/scpi)